# TerraMind Colab Benchmark

Phase 6C notebook for the frozen `terramind_v1_tiny` RGB + S1RTC validation benchmark. Do not evaluate test dates in this notebook.

Expected validated environment: Tesla T4, torch 2.11.0+cu128, numpy 2.2.6, terratorch 1.2.13, torchgeo 0.9.0.

**Completed research record:** seeds 42, 7, 123 have finished. This notebook is retained for independent reproduction; the locked benchmark uses batch size 4 and 10 epochs. Phase 7 TEST reporting is complete. Do not rerun experiments during Phase 8; see `docs/TERRAMIND_REPRODUCIBILITY.md` for completed evidence.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone or update the repo
Set `REPO_URL` to your GitHub repository URL before running.

In [ ]:
REPO_URL = 'https://github.com/uossheldon/robust-multimodal-geoai.git'
PROJECT_DIR = '/content/robust-multimodal-geoai'
import os, subprocess, pathlib
if not pathlib.Path(PROJECT_DIR).exists():
    subprocess.check_call(['git', 'clone', REPO_URL, PROJECT_DIR])
else:
    subprocess.check_call(['git', '-C', PROJECT_DIR, 'pull'])
os.chdir(PROJECT_DIR)
print('Project:', os.getcwd())

## Install TerraMind dependencies
Uses the Colab CUDA/PyTorch stack. No system CUDA install is needed.

In [ ]:
!pip install -q terratorch==1.2.13 torchgeo==0.9.0 numpy==2.2.6 "setuptools<81"

## Restore TerraMind Hugging Face cache
Uses the cached TerraMind weights from Drive when available to avoid repeated large downloads.

In [ ]:
from pathlib import Path
import subprocess
CACHE_ARCHIVE = Path('/content/drive/MyDrive/robust-multimodal-geoai/model_cache/TerraMind-1.0-tiny-hf-cache.tar.gz')
HF_CACHE = Path('/root/.cache/huggingface')
HF_CACHE.mkdir(parents=True, exist_ok=True)
if CACHE_ARCHIVE.exists():
    subprocess.check_call(['tar', '-xzf', str(CACHE_ARCHIVE), '-C', str(HF_CACHE)])
    print('Restored TerraMind HF cache from', CACHE_ARCHIVE)
else:
    print('Cache archive not found; TerraTorch may download weights once.')

from pathlib import Path
from zipfile import ZipFile
import shutil
DRIVE_ARCHIVE = Path('/content/drive/MyDrive/robust-multimodal-geoai/data/SummerSchool_Subset.zip')
local_root = Path('/content/geoai_data')
local_root.mkdir(parents=True, exist_ok=True)
archive = local_root / 'SummerSchool_Subset.zip'
if not archive.exists():
    shutil.copy2(DRIVE_ARCHIVE, archive)
extract_dir = local_root / 'SummerSchool_Subset'
extract_dir.mkdir(parents=True, exist_ok=True)
with ZipFile(archive) as zf:
    zf.extractall(extract_dir)
print('Archive:', archive)
print('Extracted:', extract_dir)


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import shutil
DRIVE_ARCHIVE = Path('/content/drive/MyDrive/robust-multimodal-geoai/data/SummerSchool_Subset.zip')
local_root = Path('/content/geoai_data')
local_root.mkdir(parents=True, exist_ok=True)
archive = local_root / 'SummerSchool_Subset.zip'
if not archive.exists():
    shutil.copy2(DRIVE_ARCHIVE, archive)
extract_dir = local_root / 'SummerSchool_Subset'
extract_dir.mkdir(parents=True, exist_ok=True)
with ZipFile(archive) as zf:
    zf.extractall(extract_dir)
print('Archive:', archive)
print('Extracted:', extract_dir)


## Verify TerraMind import and CUDA

In [ ]:
import torch, numpy as np
print('torch:', torch.__version__)
print('numpy:', np.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print('VRAM GB:', round(props.total_memory / 1024**3, 2))
from terratorch.registry import BACKBONE_REGISTRY
backbone = BACKBONE_REGISTRY.build('terramind_v1_tiny', pretrained=True, modalities=['RGB', 'S1RTC'], merge_method='mean')
print(type(backbone).__name__)
del backbone
torch.cuda.empty_cache()

## Run the frozen TerraMind benchmark
This trains only the decoder and selects the best checkpoint by validation macro mIoU.

In [ ]:
!python scripts/run_terramind.py --batch-size 4 --epochs 10 --data-root /content/geoai_data/SummerSchool_Subset

## Optional batch-size 8 run
Run only if batch size 4 is clearly safe and you intend to replace the batch-size 4 result.

In [ ]:
# !python scripts/run_terramind.py --batch-size 8 --epochs 10 --data-root /content/geoai_data/SummerSchool_Subset

## Save outputs back to Drive

In [ ]:
from pathlib import Path
import shutil
drive_root = Path('/content/drive/MyDrive/robust-multimodal-geoai')
(drive_root / 'checkpoints').mkdir(parents=True, exist_ok=True)
out = drive_root / 'colab_outputs' / 'terramind_frozen'
out.mkdir(parents=True, exist_ok=True)
if Path('results/terramind_frozen').exists():
    dest = drive_root / 'colab_outputs' / 'terramind_frozen_results'
    if dest.exists(): shutil.rmtree(dest)
    shutil.copytree('results/terramind_frozen', dest)
if Path('checkpoints/terramind_frozen_best.pt').exists():
    shutil.copy2('checkpoints/terramind_frozen_best.pt', drive_root / 'checkpoints' / 'terramind_frozen_best.pt')
for fig in Path('figures').glob('terramind_*.png'):
    shutil.copy2(fig, out / fig.name)
print('Saved TerraMind outputs under', drive_root)